In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, LSTM, Dropout, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix

from importlib import reload
import prepare_data as pda
reload(pda)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dhaaa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


<module 'prepare_data' from 'c:\\Users\\dhaaa\\OneDrive\\Skrivebord\\Skole\\Gruppe INFO284\\Info284\\prepare_data.py'>

In [25]:
data = pd.read_csv("./data/Hotel_Reviews.csv")
X, y = pda.prepare_lstm(data)

In [26]:
# Tokenize the data
tokenizer = Tokenizer(num_words = 100)
tokenizer.fit_on_texts(X)
sequences = tokenizer.texts_to_sequences(X)

# Pad the sequences
maxlen = 100
data = pad_sequences(sequences, maxlen = maxlen)

# Convert labels to numpy array
labels = np.array(y)

In [28]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, random_state=42)

In [29]:
# Build the LSTM model
model = Sequential()
model.add(Embedding(input_dim = 5000, output_dim = 128))
model.add(LSTM(units = 128, return_sequences = False))
model.add(Dropout(0.5))
model.add(Dense(units = 1, activation = 'sigmoid'))

In [35]:
# Compile the model
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs = 1, batch_size = 10000, validation_data=(X_val, y_val))

# Save the model in the recommended Keras format
model.save('./data/model/sentiment_lstm_model.keras')

print("Model training complete and saved as 'data/model/sentiment_lstm_model.keras'")

42/42 ━━━━━━━━━━━━━━━━━━━━ 145s 3s/step - accuracy: 0.9611 - loss: 0.1165 - val_accuracy: 0.9610 - val_loss: 0.1152
Model training complete and saved as 'data/model/sentiment_lstm_model.keras'


In [37]:
# Load the model
model = load_model('./data/model/sentiment_lstm_model.keras')

# Evaluate the model on the validation set
val_predictions = model.predict(X_val)
val_predictions = (val_predictions > 0.5).astype(int)

# Print classification report and confusion matrix
print(classification_report(y_val, val_predictions))
print(confusion_matrix(y_val, val_predictions))

3224/3224 ━━━━━━━━━━━━━━━━━━━━ 37s 11ms/step
              precision    recall  f1-score   support

           0       0.68      0.20      0.31      4498
           1       0.96      1.00      0.98     98650

    accuracy                           0.96    103148
   macro avg       0.82      0.60      0.64    103148
weighted avg       0.95      0.96      0.95    103148

[[  898  3600]
 [  424 98226]]


In [ ]:
# Select a subset of the data for prediction
sample_reviews = X_val[:10]  # Select the first 10 reviews for prediction
sample_labels = y_val[:10]  # Corresponding labels

# Make predictions
predictions = model.predict(sample_reviews)

# Print predictions
for review, prediction, label in zip(sample_reviews, predictions, sample_labels):
    sentiment = "Positive" if prediction > 0.5 else "Negative"
    actual_sentiment = "Positive" if label == 1 else "Negative"
    print(f"Review: {review} - Predicted Sentiment: {sentiment} - Actual Sentiment: {actual_sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
[0.99971515]
Review: [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 50 36  6  1 11 33 24 18
 50 80 76 64 37 27 62  4 30 34 38  8 38 14 80 37  4 93 11  3 23 35 28  9
 19 22 20 20 32  5 22  6  5  1 33  1 11  6  1 24 14  4 69 33 37 87  4  2
 30  1 50 51] - Predicted Sentiment: Positive - Actual Sentiment: Positive
[0.91015005]
Review: [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  8  3 27
  1  5  8 70] - Predicted Sentiment: Positive - Actual Sentiment: Positive
[0.98840064]
Review: [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0 